# Null & Missing Value Analysis
Counts nulls, empty strings, and whitespace-only values per column.

**Configure the cell below**, then **Run All**. The final cell trims whitespace and normalizes empties — run it manually.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
TABLE_NAME = "{{TABLE_NAME}}"
LAKEHOUSE_NAME = "{{LAKEHOUSE_NAME}}"
OUTPUT_SUFFIX = "_cleaned"
# ───────────────────────────────────────────────────────────────

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder.getOrCreate()
# Read from _cleaned if it exists (previous fixes), else original
try:
    df = spark.table(f"{TABLE_NAME}{OUTPUT_SUFFIX}")
    print(f"Reading from {TABLE_NAME}{OUTPUT_SUFFIX} (previous fixes applied)")
except:
    df = spark.table(TABLE_NAME)
    print(f"Reading from {TABLE_NAME} (original table)")
original_count = df.count()

print(f"Table: {TABLE_NAME}")
print(f"Rows: {original_count:,}")

In [ ]:
print("=" * 60)
print("NULL & MISSING VALUE ANALYSIS")
print("=" * 60)

null_results = []
for field in df.schema.fields:
    col_name = field.name
    null_count = df.where(F.col(col_name).isNull()).count()
    null_pct = round(null_count / original_count * 100, 2) if original_count > 0 else 0

    empty_count = 0
    whitespace_count = 0
    if isinstance(field.dataType, StringType):
        empty_count = df.where(
            (F.col(col_name).isNotNull()) & (F.col(col_name) == "")
        ).count()
        whitespace_count = df.where(
            (F.col(col_name).isNotNull()) & (F.trim(F.col(col_name)) == "")
        ).count() - empty_count

    total_missing = null_count + empty_count + whitespace_count
    total_missing_pct = round(total_missing / original_count * 100, 2) if original_count > 0 else 0

    null_results.append({
        "column": col_name,
        "type": str(field.dataType),
        "null_count": null_count,
        "null_pct": null_pct,
        "empty_string_count": empty_count,
        "whitespace_only_count": whitespace_count,
        "total_missing": total_missing,
        "total_missing_pct": total_missing_pct
    })

null_df = spark.createDataFrame(null_results)
display(null_df.orderBy(F.desc("total_missing_pct")))

high_null_cols = [r for r in null_results if r["total_missing_pct"] > 5]
if high_null_cols:
    print(f"\n⚠ {len(high_null_cols)} column(s) have >5% missing values:")
    for col_info in high_null_cols:
        print(f"  - {col_info['column']}: {col_info['total_missing_pct']}% missing")

---
## ⚠ Apply Fix — Trim Whitespace & Normalize Empties
The cell below trims whitespace from all string columns and replaces empty strings with NULL.

**Review the analysis above before running this cell.**

In [ ]:
# ⚠ APPLY FIX — Run manually after review
cleaned_df = df

string_cols = [f.name for f in cleaned_df.schema.fields if isinstance(f.dataType, StringType)]
for col_name in string_cols:
    cleaned_df = cleaned_df.withColumn(col_name, F.trim(F.col(col_name)))
    cleaned_df = cleaned_df.withColumn(
        col_name,
        F.when(F.col(col_name) == "", None).otherwise(F.col(col_name))
    )

output_table = f"{TABLE_NAME}{OUTPUT_SUFFIX}"
cleaned_df.write.mode("overwrite").format("delta").saveAsTable(
    output_table
)

print(f"Trimmed whitespace and normalized empties in {len(string_cols)} string columns")
print(f"Output: {output_table}")